# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn import metrics
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df_cancer = data.frame
df_cancer.head(5)

# Analisis del Dataset

In [ ]:
correlation = df_cancer.corr()
threshold = 0.75
filter = np.abs(correlation["target"]) > threshold
correlation_features = correlation.columns[filter].tolist()
sns.pairplot(df_cancer[correlation_features], diag_kind = "kde",  hue="target")
plt.show()

# Preprocesamiento de datos

Hacemos el Split 70-30 para train-test

In [ ]:
X = df_cancer.drop(columns="target")
y = df_cancer["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=42)

Pipeline LogisticRegression basico

In [ ]:
pl_logreg = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('classifier', LogisticRegression())
])
pl_logreg.fit(X_train, y_train)

In [ ]:
y_pred_lg = pl_logreg.predict(X_test)

# Comparamos por Curva ROC

In [ ]:
def graficarCurvaRoc( y_pred, model ):
  fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred)
  auc = metrics.roc_auc_score(y_test, y_pred)
  # Graficamos
  plt.plot(fpr,tpr,label= model +" AUC="+str(round(auc,4))) #,label= "AUC="+str(auc))
  plt.legend(loc=4, fontsize=12)
  return auc

In [ ]:
# Inicializamos los labels del gráfico
plt.figure(figsize=(20, 10))
plt.xlabel('% 1 – Specificity (falsos positivos)', fontsize=14)
plt.ylabel('% Sensitivity (positivos)', fontsize=14)

# Graficamos la recta del azar
it = [i/100 for i in range(100)]
plt.plot(it,it,label="AZAR AUC=0.5",color="black")

modelos = {'reglog':y_pred_lg}
areas = []
for pred in modelos:
    auc = graficarCurvaRoc( modelos[pred] , pred )
    areas.append( (pred, auc) )
areas = pd.DataFrame(areas, columns=['model','auc'])

# Agregamos el titulo y configuro el tamaño de letra
plt.title("Curva ROC", fontsize=14)
plt.tick_params(labelsize=12);
plt.show()

In [ ]:
areas.sort_values('auc', ascending=False)

# Ejercicios

## Analisis

### 1 - Averiguar distribución de la variable target.

Distribución de la variable target

In [ ]:
print(df_cancer['target'].value_counts())
print("\nPorcentajes:")
print(df_cancer['target'].value_counts(normalize=True) * 100)

Visualización

In [ ]:
plt.figure(figsize=(8, 5))
df_cancer['target'].value_counts().plot(kind='bar', color=['salmon', 'lightblue'])
plt.title('Distribución de la Variable Target')
plt.xlabel('Target (0=Maligno, 1=Benigno)')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.show()

### 2 - Averiguar cuales de las variables son numéricas.

In [ ]:
numerical = df_cancer.select_dtypes(include=[np.number]).columns.tolist()
print(f"Variables numéricas ({len(numerical)}):")
print(numerical)

Tipos de datos

In [ ]:
print(df_cancer.dtypes)

### 3 - Graficar Heatmap de la correlacion entre variables numericas y el target.

In [ ]:
plt.figure(figsize=(12, 10))
correlation_matrix = df_cancer[numerical].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Heatmap de Correlación - Variables Numéricas')
plt.tight_layout()
plt.show()

Correlación específica con target

In [ ]:
print("\nCorrelación con target (ordenada):")
target_corr = df_cancer[numerical].corr()['target'].sort_values(ascending=False)
print(target_corr)

## Evaluación de Modelos

### 1 - Crear Pipelines para otros modelos.

Pipeline Naive Bayes

In [ ]:
pl_nb = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('classifier', GaussianNB())
])
pl_nb.fit(X_train, y_train)
y_pred_nb = pl_nb.predict(X_test)

Pipeline SVM

In [ ]:
pl_svm = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('scaler', StandardScaler()),
    ('classifier', SVC(probability=True, random_state=42))
])
pl_svm.fit(X_train, y_train)
y_pred_svm = pl_svm.predict(X_test)

Pipeline Random Forest

In [ ]:
pl_rf = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
pl_rf.fit(X_train, y_train)
y_pred_rf = pl_rf.predict(X_test)

Pipeline Decision Tree

In [ ]:
pl_tree = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('classifier', DecisionTreeClassifier(random_state=42))
])
pl_tree.fit(X_train, y_train)
y_pred_tree = pl_tree.predict(X_test)

Pipeline KNN

In [ ]:
pl_knn = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])
pl_knn.fit(X_train, y_train)
y_pred_knn = pl_knn.predict(X_test)

### 2 - Comparar los resultados por sus curvas ROC.

In [ ]:
# Inicializamos los labels del gráfico
plt.figure(figsize=(20, 10))
plt.xlabel('% 1 – Specificity (falsos positivos)', fontsize=14)
plt.ylabel('% Sensitivity (positivos)', fontsize=14)

# Graficamos la recta del azar
it = [i/100 for i in range(100)]
plt.plot(it, it, label="AZAR AUC=0.5", color="black")

# Agregamos todos los modelos
modelos = {
    'Logistic Regression': y_pred_lg,
    'KNN': y_pred_knn,
    'Decision Tree': y_pred_tree,
    'Random Forest': y_pred_rf,
    'SVM': y_pred_svm,
    'Naive Bayes': y_pred_nb
}

areas = []
for pred in modelos:
    auc = graficarCurvaRoc(modelos[pred], pred)
    areas.append((pred, auc))

areas = pd.DataFrame(areas, columns=['model', 'auc'])

# Agregamos el titulo
plt.title("Comparación de Modelos - Curva ROC", fontsize=14)
plt.tick_params(labelsize=12)
plt.show()

Mostramos ranking de modelos

In [ ]:
print("\nRanking de Modelos por AUC:")
print(areas.sort_values('auc', ascending=False))

## Fine-tunning

### 1 - Elegir uno de los modelos y optimizarlo con Grid Search CV.

Optimizamos el modelo Random Forest (mejor desempeño esperado)

In [ ]:
# Pipeline para GridSearch
pl_rf_grid = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", ['mean concave points', 'worst radius', 'worst perimeter', 'worst concave points'])], remainder="drop")),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Parámetros a optimizar
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

# GridSearchCV con validación cruzada
grid_search = GridSearchCV(
    pl_rf_grid,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=1,
    verbose=2
)

print("Iniciando Grid Search...")
grid_search.fit(X_train, y_train)

#### Resultados de la optimización

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print(f"\nMejor AUC en validación cruzada: {grid_search.best_score_:.4f}")

# Evaluar en test
y_pred_rf_optimized = grid_search.best_estimator_.predict(X_test)
auc_optimized = metrics.roc_auc_score(y_test, y_pred_rf_optimized)
print(f"AUC en test con modelo optimizado: {auc_optimized:.4f}")

# Classification report
print("\nClassification Report (Modelo Optimizado):")
print(classification_report(y_test, y_pred_rf_optimized))

Comparación antes y después de la optimización

In [ ]:
# Curva ROC comparativa
plt.figure(figsize=(10, 6))
plt.xlabel('% 1 – Specificity (falsos positivos)', fontsize=14)
plt.ylabel('% Sensitivity (positivos)', fontsize=14)

# Recta del azar
it = [i/100 for i in range(100)]
plt.plot(it, it, label="AZAR AUC=0.5", color="black")

# Random Forest original
graficarCurvaRoc(y_pred_rf, "Random Forest (Original)")

# Random Forest optimizado
graficarCurvaRoc(y_pred_rf_optimized, "Random Forest (Optimizado)")

plt.title("Comparación: Random Forest Original vs Optimizado", fontsize=14)
plt.tick_params(labelsize=12)
plt.show()